In [0]:
from pyspark.sql.functions import col, sum as _sum, count, round as _round, avg as _avg
import traceback


In [0]:
storage_account_name = dbutils.secrets.get(scope="kv-finbank", key="datalake-account-name")
storage_account_access_key = dbutils.secrets.get(scope="kv-finbank", key="datalake-access-key")
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_path   = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"



In [0]:

print("Iniciando procesamiento Capa Gold (Agregaciones, KPIs y Optimizaciones)...")

# Rutas
delta_silver_path = f"{silver_path}"
delta_gold_path = f"{gold_path}"

try:
    # 1. LECTURA DESDE SILVER 
    df_clientes = spark.read.format("delta").load(f"{delta_silver_path}TB_CLIENTES_CORE")
    df_movimientos = spark.read.format("delta").load(f"{delta_silver_path}TB_MOV_FINANCIEROS")

    # ---------------------------------------------------------
    # REQUISITO 2: TRES TABLAS DE AGREGACIÓN
    # ---------------------------------------------------------
    
    # AGG 1: Perfil Transaccional del Cliente
    agg_perfil_cliente = df_movimientos.groupBy("id_cli") \
        .agg(
            _round(_sum("vr_mov"), 2).alias("total_monto_transaccionado"),
            count("vr_mov").alias("cantidad_movimientos")
        ).join(df_clientes.select("id_cli", "canal_adquis", "estado_cli"), "id_cli", "inner")

    # AGG 2: Rendimiento por Canal de Adquisición
    agg_canal_adquisicion = agg_perfil_cliente.groupBy("canal_adquis") \
        .agg(
            count("id_cli").alias("total_clientes_activos"),
            _round(_sum("total_monto_transaccionado"), 2).alias("volumen_monetario_canal")
        )

    # AGG 3: Resumen Temporal Operativo 
    agg_resumen_temporal = df_movimientos.groupBy("ingest_year", "ingest_month") \
        .agg(
            count("id_cli").alias("volumen_transacciones_mensual"),
            _round(_sum("vr_mov"), 2).alias("flujo_dinero_mensual")
        )

    # ---------------------------------------------------------
    # REQUISITO 5: TABLA DE KPIs EJECUTIVOS
    # ---------------------------------------------------------
    # Consolida las métricas globales para un dashboard gerencial rápido (Single Row Table)
    kpi_ejecutivo = df_movimientos.agg(
        _round(_sum("vr_mov"), 2).alias("kpi_volumen_total_procesado"),
        count("vr_mov").alias("kpi_total_transacciones"),
        _round(_avg("vr_mov"), 2).alias("kpi_ticket_promedio")
    ).crossJoin(
        df_clientes.filter(col("estado_cli") == "Activo").agg(count("id_cli").alias("kpi_clientes_activos_totales"))
    )

    # ---------------------------------------------------------
    # REQUISITO 3: ESCRITURA CON PARTICIONAMIENTO Y OPTIMIZACIÓN (Z-ORDER)
    # ---------------------------------------------------------
    print("Escribiendo y optimizando modelos en Gold...")

    # Guardar AGG 1 (Particionado por estado_cli, es cardinalidad baja)
    ruta_perfil = f"{delta_gold_path}AGG_PERFIL_CLIENTE"
    agg_perfil_cliente.write.format("delta").mode("overwrite").partitionBy("estado_cli").save(ruta_perfil)
    # Clustering (Z-ORDER) por canal_adquis para consultas más rápidas
    spark.sql(f"OPTIMIZE delta.`{ruta_perfil}` ZORDER BY (canal_adquis)")

    # Guardar AGG 2 (Tabla pequeña, sin partición, solo overwrite)
    agg_canal_adquisicion.write.format("delta").mode("overwrite").save(f"{delta_gold_path}AGG_CANAL_ADQUISICION")

    # Guardar AGG 3 (Particionado por año)
    ruta_temporal = f"{delta_gold_path}AGG_RESUMEN_TEMPORAL"
    agg_resumen_temporal.write.format("delta").mode("overwrite").partitionBy("ingest_year").save(ruta_temporal)
    
    # Guardar KPI Ejecutivo
    kpi_ejecutivo.write.format("delta").mode("overwrite").save(f"{delta_gold_path}KPI_EJECUTIVO_FINANCIERO")

    print("ÉXITO: 3 Tablas de agregación y 1 Tabla de KPIs creadas y optimizadas.")
    print("SUCCESS")

except Exception as e:
    error_msg = f"Falla crítica en el pipeline Gold: {str(e)}"
    print(error_msg)
    traceback.print_exc()
    print(f"FAILED: {error_msg}")

Iniciando procesamiento Capa Gold (Agregaciones, KPIs y Optimizaciones)...
Escribiendo y optimizando modelos en Gold...
✅ ÉXITO: 3 Tablas de agregación y 1 Tabla de KPIs creadas y optimizadas.
SUCCESS


In [0]:
from pyspark.sql.functions import sum as _sum, round as _round

print("--- VALIDACIÓN DE LA CAPA GOLD ---")

# 1. Validar que las tablas existen y tienen datos
df_kpi = spark.read.format("delta").load(f"{gold_path}KPI_EJECUTIVO_FINANCIERO")
print("\n1. Vista Previa de la Tabla KPI Ejecutivo:")
df_kpi.show(truncate=False)

# 2. Validar que la optimización (Z-ORDER / Particiones) se aplicó
print("2. Verificando metadatos de optimización de AGG_PERFIL_CLIENTE:")
display(spark.sql(f"DESCRIBE DETAIL delta.`{gold_path}AGG_PERFIL_CLIENTE`").select("format", "partitionColumns"))

# 3. PRUEBA DE RECONCILIACIÓN (El equivalente al DQ en Gold)
print("\n3. Prueba de Reconciliación Matemática (Silver vs Gold):")

# Cuánto dinero hay en la capa transaccional limpia (Silver)
total_silver = spark.read.format("delta").load(f"{silver_path}TB_MOV_FINANCIEROS") \
    .agg(_round(_sum("vr_mov"), 2)).collect()[0][0]

# Cuánto dinero reporta nuestro dashboard gerencial (Gold)
total_gold = df_kpi.select("kpi_volumen_total_procesado").collect()[0][0]

print(f"Total procesado en Silver: {total_silver}")
print(f"Total consolidado en Gold : {total_gold}")

if total_silver == total_gold:
    print("RECONCILIACIÓN EXITOSA: La agregación es matemáticamente exacta. No hay pérdida de datos.")
else:
    print("ALERTA DE CUADRE: Las cifras no coinciden. Revisar la lógica de agrupación.")

--- VALIDACIÓN DE LA CAPA GOLD ---

1. Vista Previa de la Tabla KPI Ejecutivo:
+---------------------------+-----------------------+-------------------+----------------------------+
|kpi_volumen_total_procesado|kpi_total_transacciones|kpi_ticket_promedio|kpi_clientes_activos_totales|
+---------------------------+-----------------------+-------------------+----------------------------+
|1.23994386729571E12        |495045                 |2504709.4          |8500                        |
+---------------------------+-----------------------+-------------------+----------------------------+

2. Verificando metadatos de optimización de AGG_PERFIL_CLIENTE:


format,partitionColumns
delta,List(estado_cli)



3. Prueba de Reconciliación Matemática (Silver vs Gold):
💰 Total procesado en Silver: 1239943867295.71
💰 Total consolidado en Gold : 1239943867295.71
✅ RECONCILIACIÓN EXITOSA: La agregación es matemáticamente exacta. No hay pérdida de datos.
